## Part 3 - Pre-training, Fine-tuning, LoRA/QLoRA, RLHF, Precision Formats & Distributed Training

**Covered:** Pre-training objectives and scaling laws (Chinchilla), data curation pipeline, catastrophic forgetting, full fine-tuning mitigations, LoRA (complete mathematical derivation, rank selection, merging), QLoRA (NF4, double quantization, paged optimizers, hardware requirements), RLHF (SFT, reward modeling, PPO with KL penalty, alignment tax), DPO, floating point formats (FP16 failure modes, BF16 design, FP8), ZeRO Stages 1/2/3, tensor parallelism, pipeline parallelism, 3D parallelism, PyTorch training pitfalls.

In [ ]:
# 15. Pre-training: The Foundation

## 15. Pre-training: The Foundation

### 15.1 The Pre-training Objective

The canonical pre-training objective for decoder-only models is **Causal Language Modeling (CLM)**, also called next-token prediction. Given a sequence of tokens $x_1, x_2, \ldots, x_n$, the model is trained to maximize:

$$\mathcal{L}_{\text{CLM}} = -\sum_{t=1}^{n} \log P(x_t \mid x_1, \ldots, x_{t-1}; \theta)$$

This is the negative log-likelihood of the observed sequence. Minimizing it is equivalent to maximizing the probability the model assigns to the actual next token.

**Why this objective is remarkably powerful:** It is **self-supervised** — every token in every document is simultaneously a training input and a supervision signal. A 1 trillion token corpus yields exactly 1 trillion gradient signals, with zero human labeling cost. The diversity of the objective forces generalization: the same weights must predict the next word in Python code, a legal contract, a medical paper, and a Shakespearean sonnet. The only way to minimize loss across all these domains simultaneously is to build genuinely general representations.

**Perplexity as the training metric:**

$$\text{PPL} = \exp\!\left(\frac{1}{n}\sum_{t=1}^n -\log P(x_t \mid x_{<t})\right)$$

Perplexity is the exponentiated average negative log-likelihood per token. A perplexity of $k$ means the model is, on average, as uncertain as if it were choosing uniformly among $k$ equally likely options. State-of-the-art models achieve single-digit perplexity on held-out web text. **Crucially, perplexity alone does not predict downstream task performance** — this is the fundamental limitation of pre-training metrics.

---

### 15.2 Scaling Laws: The Chinchilla Result

**Kaplan et al. (2020)** established that model loss scales predictably with compute $C$, dataset size $D$, and parameters $N$:

$$\mathcal{L}(N, D) \approx \frac{A}{N^\alpha} + \frac{B}{D^\beta} + \mathcal{L}_\infty$$

where $\mathcal{L}_\infty$ is the irreducible loss (entropy of natural language). This suggested that for a fixed compute budget, model size should grow much faster than data size — leading to the era of large but undertrained models (GPT-3 at 300B tokens, 175B parameters).

**Hoffmann et al. (2022) — the Chinchilla correction:** A more careful empirical study found that **compute-optimal training requires equal scaling of parameters and tokens**: for every doubling of model size, you should also double the training data. The optimal ratio is approximately:

$$D_{\text{optimal}} \approx 20 \times N$$

Chinchilla (70B parameters, 1.4T tokens) outperformed Gopher (280B parameters, 300B tokens) on almost all benchmarks, using the same compute budget. This result fundamentally changed how the field approaches pre-training: Llama-1's 7B model was trained on 1T tokens (far beyond the Chinchilla optimum for inference efficiency), Llama-3's 8B model on 15T tokens.

**The inference-optimal vs. compute-optimal tension:** Chinchilla optimal is compute-optimal for a single training run. But for deployment, you want the **smallest model** that achieves a given quality level, because inference costs dominate at scale. Training a smaller model on more data (compute-suboptimal) produces a model that is cheaper to serve while achieving similar quality — this is the Llama philosophy.

---

### 15.3 Data Curation: The Unsung Determinant of Quality

The quality of pre-training data dominates model quality at matched compute. Common pipeline stages:

**1. Deduplication:** Near-duplicate documents cause the model to memorize rather than generalize. MinHash LSH is standard for fuzzy deduplication at web scale. Exact deduplication with suffix arrays (Lee et al., 2022) catches verbatim copies. Llama-3's data pipeline includes both exact and fuzzy dedup.

**2. Quality filtering:** Heuristic filters (perplexity on a reference model, character-to-token ratio, fraction of alphabetic characters) remove machine-generated spam and low-quality text. Classifier-based filtering (train a fastText classifier on curated vs. web data) is more powerful but computationally expensive.

**3. Harmful content removal:** Classifier-based removal of CSAM, violent content, and PII. URL blocklists for known harmful domains. This is both ethical and practical — training on harmful content degrades model alignment.

**4. Domain mixing:** The ratio of web text, books, code, scientific papers, and curated sources is a critical hyperparameter. Code data dramatically improves reasoning ability even on non-code tasks (hypothesis: code teaches the model sequential, causal, step-by-step reasoning). Llama-3's mix included significantly more code than Llama-2.

**5. Data ordering and curriculum:** Training on easier/shorter sequences first, then harder/longer ones can improve convergence, analogous to curriculum learning in supervised settings. Most large-scale training uses shuffled data (no curriculum) for simplicity, but some models experiment with structured curricula.

---


In [ ]:
# 16. Fine-tuning: Adapting the Pre-trained Foundation

## 16. Fine-tuning: Adapting the Pre-trained Foundation

### 16.1 The Catastrophic Forgetting Problem

A pre-trained base model is a general-purpose next-token predictor. It is not an assistant. When you fine-tune aggressively on a narrow dataset (e.g., customer service transcripts), gradient descent optimizes the weights for the new task. Since gradient descent doesn't distinguish "useful general knowledge" from "inefficient parameters to update," it can overwrite general capabilities.

**Formally:** Let $\theta^*$ be the pre-trained weights minimizing $\mathcal{L}_{\text{pre-train}}(\theta)$ and $\theta'$ be the fine-tuned weights minimizing $\mathcal{L}_{\text{fine-tune}}(\theta)$. If the fine-tuning distribution is narrow and the learning rate is high, $\theta'$ can be far from $\theta^*$ in weight space, with $\mathcal{L}_{\text{pre-train}}(\theta') \gg \mathcal{L}_{\text{pre-train}}(\theta^*)$. The model has "forgotten" its pre-training.

**Mitigations in full fine-tuning:**
- **Low learning rate:** $\eta \sim 10^{-5}$ to $10^{-6}$ vs. $10^{-3}$ to $10^{-4}$ for pre-training
- **Distribution mixing:** Keep 10-30% pre-training data in the fine-tuning mix to anchor weights
- **Elastic Weight Consolidation (EWC):** Add a regularization term $\lambda \sum_i F_i (\theta_i - \theta_i^*)^2$ where $F_i$ is the Fisher information — the importance of each parameter to the pre-training objective. Penalizes moving important parameters. Computationally expensive at LLM scale.
- **LoRA:** The practical solution — freeze the base model entirely, train only small adapter matrices

---


### 16.2 LoRA: Low-Rank Adaptation — Full Derivation

**Hu et al., 2021.** The central hypothesis: **the weight updates required to adapt a pre-trained model to a new task have a low intrinsic dimensionality** — they lie in a low-dimensional subspace of the full parameter space.

**Mathematical basis for this hypothesis:** Aghajanyan et al. (2020) showed that fine-tuning objectives can be minimized by optimizing in subspaces of dimension as low as 100-1000, regardless of the full parameter count. The pre-trained model already spans the relevant representation space; fine-tuning is a small directional adjustment within it.

**The LoRA construction:**

For a weight matrix $W_0 \in \mathbb{R}^{m \times n}$ (frozen), instead of learning a full $\Delta W \in \mathbb{R}^{m \times n}$, we constrain the update to rank $r$:

$$\Delta W = BA$$

where $B \in \mathbb{R}^{m \times r}$, $A \in \mathbb{R}^{r \times n}$, with $r \ll \min(m, n)$.

The modified forward pass:

$$h = W_0 x + \Delta W x = W_0 x + BA x$$

**Initialization:**
- $A$ is initialized with random Gaussian values: $A \sim \mathcal{N}(0, \sigma^2)$
- $B$ is initialized to **zero**: $B = 0$

This guarantees $\Delta W = BA = 0$ at the start of training, meaning LoRA begins exactly at the pre-trained weights. Training starts from a stable baseline.

**The scaling factor $\alpha$:**

In practice, the update is scaled:
$$h = W_0 x + \frac{\alpha}{r} BA x$$

$\alpha$ is a fixed hyperparameter (not trained). Setting $\alpha = r$ gives $\alpha/r = 1$ — equivalent to no scaling. The scaling factor allows tuning the magnitude of the LoRA contribution independently of $r$. Common practice: set $\alpha = 2r$ or $\alpha = 16$ while varying $r$.

**Parameter reduction analysis:**

For a single attention projection $W^Q \in \mathbb{R}^{4096 \times 4096}$:
- Full fine-tuning trainable parameters: $4096^2 = 16,777,216$
- LoRA with $r = 8$: $4096 \times 8 + 8 \times 4096 = 65,536$
- **Reduction factor: 256×**

For a full model with LoRA applied to all attention projections ($W^Q, W^K, W^V, W^O$ in each of $L = 32$ layers):
$$\text{LoRA params} = 4 \times 2 \times d_{model} \times r \times L = 4 \times 2 \times 4096 \times 8 \times 32 = 8,388,608$$

A 7B parameter model fine-tuned with LoRA trains only ~8M parameters — **0.12% of total parameters**.

**Selecting $r$:**

- $r = 4$: Minimal adaptation, very fast, suitable for style/format changes
- $r = 8$: The practical default for most instruction fine-tuning
- $r = 16$ to $r = 64$: For complex domain adaptation (medical, legal, code)
- $r = 256$+: Approaches full fine-tuning expressivity; rarely justified

**Which matrices to apply LoRA to:**

Original paper: $W^Q$ and $W^V$ only. Subsequent work found applying to all four attention projections ($W^Q, W^K, W^V, W^O$) and even FFN layers ($W_1, W_2$) consistently improves quality. The tradeoff is more trainable parameters.

**Merging at inference:**

After training, the LoRA matrices can be merged back into the base weights:
$$W_{\text{merged}} = W_0 + \frac{\alpha}{r} BA$$

This adds **zero inference overhead** — the merged model is identical in architecture to the base model. No adapter layers, no additional FLOPs. This is a major practical advantage over other PEFT approaches that require permanent adapter layers at inference time.

---

### 16.3 QLoRA: Quantized LoRA — Democratizing Fine-tuning

**Dettmers et al., 2023.** Combines three techniques to enable fine-tuning of 65B+ parameter models on a single consumer GPU:

**Technique 1: 4-bit NormalFloat (NF4) Quantization**

Standard quantization maps a range of values to $2^k$ discrete levels. For 4-bit: 16 discrete levels. The choice of where to place those 16 levels matters enormously.

NF4 uses a **quantile-based placement**: the 16 quantile boundaries of a standard normal distribution $\mathcal{N}(0, 1)$. Pre-trained model weights are empirically approximately normally distributed (a consequence of random initialization and gradient descent). NF4 places quantization levels where the data actually is, minimizing quantization error.

$$q_i = \frac{1}{2}\left(Q_X^{-1}\!\left(\frac{i}{2^k + 1}\right) + Q_X^{-1}\!\left(\frac{i+1}{2^k + 1}\right)\right)$$

where $Q_X^{-1}$ is the quantile function of the empirical weight distribution.

**Comparison to standard int4:** Standard uniform int4 places levels at equal intervals, wasting quantization capacity in the tails where few weights exist. NF4 concentrates levels where weights are dense (near zero), significantly reducing quantization error for normally-distributed weights.

**Technique 2: Double Quantization**

Quantization requires storing quantization constants (scale factors and zero points) for each block of weights. These constants themselves consume memory. QLoRA quantizes the quantization constants — "quantizing the quantizer."

For a blocksize of 64 weights with FP32 quantization constants: each constant takes 32 bits for 64 weights = 0.5 bits/parameter overhead. After double quantization (to 8-bit constants with blocksize 256): 8/256 = 0.03 bits/parameter overhead. **Net reduction: 0.47 bits/parameter** — small but meaningful at scale.

**Technique 3: Paged Optimizers**

NVIDIA unified memory allows CPU RAM to act as overflow storage for GPU memory. During training, optimizer states (the large Adam moment matrices) are stored in CPU RAM when GPU memory is tight and paged into GPU memory only during the weight update step. This allows brief memory spikes during backward pass without OOM crashes.

**Hardware requirement comparison (7B model):**

| Method | Precision | VRAM Required |
|---|---|---|
| Full fine-tuning | FP16 | ~112 GB (model + grad + optimizer) |
| LoRA | FP16 | ~28 GB (frozen model + grad + small optimizer state) |
| QLoRA | NF4 + FP16 adapters | ~6-8 GB |

QLoRA puts 7B model fine-tuning on an RTX 3090 (24 GB VRAM). 13B models on an A6000 (48 GB). **This is the democratization event for LLM development.**

**The quality cost of QLoRA:** Dettmers et al. showed QLoRA fine-tuned models match full 16-bit fine-tuning performance on most benchmarks. The NF4 quantization loss is compensated by LoRA's targeted adaptation. For most practical fine-tuning tasks, the quality difference is negligible.

---

In [ ]:
# 17. RLHF: Alignment From Human Feedback

## 17. RLHF: Alignment From Human Feedback

### 17.1 Why CLM-trained Models Need Alignment



A base model trained on CLM is a **document continuation engine**. Given "How do I make a bomb?", the statistically likely continuation in training data might be an actual explanation (from chemistry textbooks, fictional novels, security research papers). The model has no concept of "this is harmful to output."

More subtly, instruction-following emerges weakly from CLM. The model can follow instructions in-context (because instructions followed by responses are common in training data), but it's unreliable and inconsistent.

**The alignment goal:** Make the model reliably helpful, harmless, and honest (HHH) — not by restricting its knowledge, but by changing its behavioral policy to produce outputs that humans prefer.

---

### 17.2 The Three-Stage RLHF Pipeline

**Stage 1: Supervised Fine-tuning (SFT)**

Collect high-quality demonstration data: human-written responses to diverse prompts. Fine-tune the base model on this data with standard CLM loss. This creates the **SFT model** — a well-behaved starting point for RL training.

SFT alone produces significant improvement in instruction following. InstructGPT (Ouyang et al., 2022) showed that 1.3B parameter SFT models outperform 175B base models on human preference evaluations — alignment quality matters more than raw scale for user-facing tasks.

**Stage 2: Reward Model Training**

Collect **comparison data**: human raters evaluate multiple model responses to the same prompt and rank them by preference. The reward model $r_\phi(x, y)$ is trained to predict human preference from (prompt, response) pairs.

Training objective: For a prompt $x$ with preferred response $y_w$ and rejected response $y_l$:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}_{(x, y_w, y_l) \sim D}\!\left[\log \sigma\!\left(r_\phi(x, y_w) - r_\phi(x, y_l)\right)\right]$$

This is the **Bradley-Terry preference model** — it trains the reward model to assign higher scores to preferred responses. $\sigma$ is the sigmoid function; the loss approaches 0 when $r_\phi(x, y_w) \gg r_\phi(x, y_l)$.

**Reward model architecture:** Typically the SFT model with the final token prediction head replaced by a scalar output head. The scalar is the reward for the entire response.

**The reward hacking problem (Goodhart's Law):** The reward model is an imperfect proxy for human preference. As RL optimization pressure increases, the policy learns to exploit reward model weaknesses — producing responses that score high on the proxy but are not actually preferred by humans. Classic examples: extremely verbose responses (length bias in annotators), excessive hedging, sycophantic agreement with stated user preferences.

**Stage 3: RL Fine-tuning with PPO**

Use the reward model as the reward signal and optimize the SFT model (policy) using **Proximal Policy Optimization (PPO)**:

$$\mathcal{L}_{\text{PPO}} = \mathbb{E}_{(x,y) \sim \pi_\theta}\!\left[r_\phi(x, y) - \beta \cdot D_{\text{KL}}\!\left[\pi_\theta(\cdot \mid x) \;\|\; \pi_{\text{ref}}(\cdot \mid x)\right]\right]$$

**The KL divergence penalty** is critical:

$$D_{\text{KL}}[\pi_\theta \| \pi_{\text{ref}}] = \sum_y \pi_\theta(y \mid x) \log \frac{\pi_\theta(y \mid x)}{\pi_{\text{ref}}(y \mid x)}$$

where $\pi_{\text{ref}}$ is the frozen SFT model. This term penalizes the policy for drifting too far from the SFT model.

**Why the KL penalty is essential:**
- Without it, PPO can collapse the policy to generate only the handful of response patterns that happen to score high on the reward model (mode collapse)
- The KL penalty keeps the policy in the neighborhood of the SFT model, preserving diversity and pre-trained knowledge
- It prevents reward hacking from escalating to complete gibberish

**The $\beta$ tradeoff:** Large $\beta$ → policy stays close to SFT, minimal reward maximization. Small $\beta$ → aggressive reward maximization, risk of reward hacking and capability degradation. Tuning $\beta$ is one of the critical RLHF hyperparameters. Typical values: $0.1$ to $0.5$.

---

### 17.3 The Alignment Tax

Empirical finding across many models: RLHF alignment consistently reduces performance on certain capability benchmarks — particularly open-ended generation, creative tasks, and tasks requiring the model to reason through unconventional solution paths.

**The mechanism:** PPO with KL penalty effectively shrinks the model's output distribution toward a safe, conventional mode. The model learns to prefer high-reward, safe, predictable responses. Low-probability but potentially brilliant or creative token sequences get systematically suppressed because they're risky — they might score poorly on the reward model.

**The intellectual timidity problem:** Aligned models tend to hedge, qualify, and default to "I'm not sure" more than base models. They avoid making confident claims even when the base model would be correct. This is the reward model's length and hedging bias being exploited.

**DPO as an alternative (Rafailov et al., 2023):** Direct Preference Optimization bypasses the separate reward model entirely. It shows that the optimal RLHF policy has a closed-form solution that can be expressed as a classification objective directly on preference pairs:

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}\!\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w \mid x)}{\pi_{\text{ref}}(y_w \mid x)} - \beta \log \frac{\pi_\theta(y_l \mid x)}{\pi_{\text{ref}}(y_l \mid x)}\right)\right]$$

DPO is simpler to implement (no RL, no reward model), more stable to train, and produces comparable or better results. Most modern alignment pipelines use DPO or variants (IPO, KTO) rather than PPO.

---

In [ ]:
# 18. Numerical Precision: FP32, FP16, BF16, FP8

## 18. Numerical Precision: FP32, FP16, BF16, FP8


### 18.1 IEEE 754 Floating Point Formats

A floating point number is represented as:

$$(-1)^s \times 2^{e - \text{bias}} \times (1 + \text{mantissa})$$

| Format | Total bits | Sign | Exponent | Mantissa | Max value | Min positive |
|---|---|---|---|---|---|---|
| FP32 | 32 | 1 | 8 | 23 | $\sim 3.4 \times 10^{38}$ | $\sim 1.2 \times 10^{-38}$ |
| FP16 | 16 | 1 | 5 | 10 | $65,504$ | $\sim 6.1 \times 10^{-5}$ |
| BF16 | 16 | 1 | 8 | 7 | $\sim 3.4 \times 10^{38}$ | $\sim 1.2 \times 10^{-38}$ |
| FP8 (E4M3) | 8 | 1 | 4 | 3 | $448$ | $\sim 0.002$ |

---

### 18.2 Why FP16 Fails for Deep Learning

FP16's 5-bit exponent gives a maximum representable value of $65,504$. In deep neural networks:

**Overflow:** Intermediate activations and gradients during backward pass can easily exceed 65,504, especially in deep networks with large batch sizes. When overflow occurs, the value becomes `Inf`. Once `Inf` enters the computation graph, it propagates: `Inf - Inf = NaN`. The training run is dead.

**Underflow:** Very small gradients (which occur naturally in deep networks) can fall below the FP16 minimum positive value of ~$6 \times 10^{-5}$, rounding to exactly zero. The gradient signal vanishes at that parameter permanently.

**The loss scaling hack:** To use FP16, practitioners multiplied the loss by a large scalar (e.g., 1024 or 32768) before backward, scaling all gradients up into the FP16 representable range. Then divide by the same scalar before the weight update. This required:
- Monitoring for overflow (NaN/Inf) and reducing the scale factor when it occurs
- Re-running the step if overflow was detected
- Complex infrastructure to manage the dynamic scaling factor

Fragile, slow, and required careful tuning. Any overflow detection failure corrupted the training run.

---

### 18.3 BF16: The Elegant Fix

**Brain Float 16 (Google Brain, 2018).** The key insight: for neural network training, **dynamic range matters more than precision**. The model needs to represent very large numbers (forward pass activations) and very small numbers (gradients) without overflow or underflow. It doesn't need 10 bits of mantissa precision — 7 bits is sufficient for stochastic gradient descent to converge.

BF16 allocates 8 bits to the exponent (same as FP32) and only 7 bits to the mantissa. This gives **FP32 dynamic range in 16 bits**, completely eliminating overflow and underflow as practical concerns. Loss scaling becomes unnecessary.

**The precision tradeoff:** BF16 has $2^7 = 128$ discrete levels between powers of 2, vs. FP32's $2^{23} = 8,388,608$. The numbers are "blurrier." But SGD-based optimizers don't need high precision — they're averaging over many stochastic samples anyway. The direction of the gradient matters more than its precise magnitude. Empirically, BF16 training matches FP32 quality while using half the memory and achieving higher throughput (hardware can perform more BF16 ops/second than FP32).

**Mixed precision training (standard pattern):**
- Model weights: BF16 (forward pass, backward pass)
- Optimizer states (Adam moments): FP32 (weight updates require precision to accumulate small incremental changes over many steps)
- Gradients: BF16 during backward, converted to FP32 for optimizer step

This is the default in PyTorch's `torch.cuda.amp` with `dtype=torch.bfloat16`.

---
### 18.4 FP8: The Frontier (H100 and Beyond)

NVIDIA H100 introduced hardware FP8 support (E4M3 and E5M2 formats). FP8 halves the memory of BF16 and doubles throughput for matrix multiplications. Used for **inference quantization** and experimentally for training (Transformer Engine in Hopper architecture).

FP8 training requires careful per-tensor scaling to prevent the extremely limited range (max 448 in E4M3) from causing overflow in activations. NVIDIA's Transformer Engine handles this automatically using calibrated scaling factors per operation.

---

In [ ]:
# 19. Distributed Training: ZeRO and Model Parallelism

## 19. Distributed Training: ZeRO and Model Parallelism

### 19.1 The Memory Wall

For a 70B parameter model in BF16:
- Model weights: $70 \times 10^9 \times 2 \text{ bytes} = 140 \text{ GB}$
- Gradients (same size): $140 \text{ GB}$
- Adam optimizer states (FP32): $70 \times 10^9 \times 2 \times 4 \text{ bytes} = 560 \text{ GB}$
- **Total: ~840 GB**

An H100 has 80 GB HBM. A single GPU cannot hold even the weights, let alone the full training state. Distributed training is mandatory.

---

### 19.2 Data Parallel Training (Baseline)

Each GPU holds a **complete copy** of the model. The dataset is partitioned across GPUs. Each GPU computes gradients on its local data shard, then gradients are **all-reduced** (summed and averaged) across all GPUs using collective communication (NCCL's ring all-reduce).

**Memory issue:** Every GPU holds the full model + full gradients + full optimizer state. For 70B: 840 GB per GPU. Impossible.

**Compute issue:** Communication of gradients ($140 \text{ GB}$) must happen every step. On 8 GPUs over NVLink (600 GB/s): $140 / 600 \approx 0.23$ seconds per step just for communication. This becomes a bottleneck at scale.

---

### 19.3 ZeRO: Zero Redundancy Optimizer

**Rajbhandari et al. (DeepSpeed, 2020).** ZeRO eliminates the redundant copies of model state across data-parallel processes. Three stages of increasing aggressiveness:

**ZeRO Stage 1 — Optimizer State Partitioning:**

Instead of every GPU holding all Adam states, GPU $k$ holds only the optimizer states for $\frac{1}{N}$ of the parameters (its assigned partition). During the optimizer step, each GPU updates only its shard. A subsequent all-gather reconstructs the full updated weights.

$$\text{Memory per GPU} = \frac{1}{N}\left(12 \text{ bytes/param}\right) + 4 \text{ bytes/param (weights)} + 2 \text{ bytes/param (gradients)}$$

For $N=64$ GPUs: optimizer state per GPU reduces from 560 GB to 8.75 GB. **Total per GPU: ~18 GB for 70B model.**

**ZeRO Stage 2 — Gradient Partitioning:**

Additionally, each GPU only stores gradients for its parameter partition. Gradients are reduced directly into the responsible GPU's shard rather than all-reducing the full gradient tensor.

$$\text{Memory per GPU} \approx \frac{14 \times \text{bytes/param}}{N} + 2 \text{ bytes/param (weights)}$$

For $N=64$: each GPU needs approximately **2.2 GB for optimizer+gradients + 140 GB weights** ... but the weights themselves still need to be replicated for the forward/backward pass. Stage 2 alone gets you:

**ZeRO Stage 3 — Parameter Partitioning:**

The most radical stage. Each GPU holds only $\frac{1}{N}$ of the model weights. No GPU has the full model at any time.

$$\text{Memory per GPU} = \frac{16 \text{ bytes/param}}{N}$$

For $N=64$: **$\approx 840 / 64 \approx 13.1 \text{ GB per GPU}$**. A 70B model fits on 64 GPUs with a single A100 (40 GB) per GPU — feasible.

**The communication cost of Stage 3:**

During the forward pass, as computation proceeds layer by layer, each GPU must **all-gather** the parameters for the current layer from all other GPUs. After computing that layer, the gathered parameters are immediately **discarded** from local memory (to free space for the next layer). During backward, the same all-gather + discard pattern occurs, followed by a **reduce-scatter** of gradients into the owning GPU's partition.

This doubles the communication volume compared to Stage 1/2, but the communication can be **overlapped with computation** (pre-fetching the next layer's parameters while computing the current layer). With NVLink (600 GB/s) or InfiniBand (200 Gb/s), this overlap makes Stage 3 practical.

**ZeRO-Infinity:** Extends Stage 3 to offload parameters and optimizer states to CPU RAM and even NVMe SSDs when GPU memory is exhausted. Enables trillion-parameter model training on GPU clusters with aggressive CPU offloading.

---

### 19.4 Tensor Parallelism (Megatron-LM)

ZeRO is a data parallelism optimization. **Tensor Parallelism** partitions individual weight matrices across devices.

For an FFN layer $Y = \text{GeLU}(XA)B$ with $A \in \mathbb{R}^{d \times 4d}$, $B \in \mathbb{R}^{4d \times d}$:

**Column-parallel (for A):** Partition $A$ by columns: $A = [A_1, A_2]$ where each half is on a different GPU. Each GPU computes $Y_i = \text{GeLU}(XA_i)$ independently (no communication). **Row-parallel (for B):** Partition $B$ by rows: $B = [B_1; B_2]$ where $Y = Y_1 B_1 + Y_2 B_2$. Each GPU computes its contribution; a single **all-reduce** sums the results.

Result: one all-reduce per FFN layer (instead of per-parameter communication). Much more communication-efficient than ZeRO Stage 3 for this case.

**Attention is tensor-parallelized** by partitioning attention heads across GPUs — each GPU owns a subset of heads. No communication needed within the attention computation; one all-reduce after the output projection.

---

### 19.5 Pipeline Parallelism

For models too large even for tensor parallelism across available devices, **pipeline parallelism** partitions the model by layer depth: GPU 1 holds layers 1-10, GPU 2 holds layers 11-20, etc. The forward pass "pipelines" microbatches through the stages.

**The bubble problem:** Naive pipeline parallelism (GPipe) has a pipeline bubble — GPU 1 is idle during the backward pass while GPU 4 finishes its forward pass. Bubble fraction: $(p-1)/(m+p-1)$ where $p$ is pipeline depth and $m$ is number of microbatches. With $p=8$ stages and $m=32$ microbatches: $(8-1)/(32+8-1) = 7/39 \approx 18\%$ idle time.

**Interleaved scheduling (Megatron-LM v2)** assigns non-contiguous layers to each GPU (e.g., GPU 1 has layers 1-5 and 41-45) using virtual stages, reducing the bubble fraction at the cost of more communication.

**3D Parallelism:** Production LLM training (e.g., Megatron-Turing NLG 530B, GPT-4) combines all three: Data Parallelism (ZeRO) + Tensor Parallelism + Pipeline Parallelism. The 3D combination allows scaling to thousands of GPUs while maintaining efficiency.

---

In [ ]:
# 20. PyTorch Training Hygiene: Silent Failure Modes

## 20. PyTorch Training Hygiene: Silent Failure Modes


These are issues that experienced engineers still encounter and that can waste days of debugging.

**`model.train()` vs `model.eval()`:**
- `model.train()`: Dropout is active (randomly zeroing activations for regularization), BatchNorm uses batch statistics
- `model.eval()`: Dropout is disabled (full network used), BatchNorm uses running statistics
- **Failure mode:** Evaluating with `model.train()` gives noisy, unreproducible metrics. Training with `model.eval()` disables dropout regularization, leading to overfitting without warning. Always set mode explicitly.

**`optimizer.zero_grad()`:**
- PyTorch **accumulates** gradients by default. If you don't zero gradients before each backward pass, you're adding gradients from multiple batches together
- **Failure mode:** Loss appears to decrease slowly or erratically. Gradient norms grow unexpectedly. The bug is silent — no error, just wrong behavior
- **Efficient pattern:** `optimizer.zero_grad(set_to_none=True)` sets gradients to `None` rather than zero tensors, saving memory allocation

**`torch.no_grad()` during inference:**
- Without this context manager, PyTorch builds the computation graph (autograd tape) even during inference, consuming memory proportional to the forward pass
- **Failure mode:** OOM during inference that shouldn't occur given the model size

**Gradient accumulation:**
For large effective batch sizes that don't fit in GPU memory, accumulate gradients over $k$ forward-backward passes before updating weights:
```python
for i, batch in enumerate(dataloader):
    loss = model(batch) / accumulation_steps
    loss.backward()  # accumulates into .grad
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```
**Failure mode:** Forgetting to divide loss by `accumulation_steps` scales the effective learning rate by $k$, causing training instability.

**Random seed management in distributed training:**
Each process must have the same model initialization seed but different data sampling seeds. Failure to set different data seeds leads to all GPUs training on identical data — effectively a $1/N$ reduction in effective dataset size.